# Giovanni Time Series Adapter Regression Test Suite

This notebook contains a regression test for the Giovanni Time Series adapter to ensure the service continues function with all Harmony updates.

## Prerequisites

The dependencies for this notebook are listed in the [environment.yaml](./environment.yaml). To test or install locally, create the papermill environment used in the automated regression testing suite:

`conda env create -f ./environment.yaml && conda activate papermill-giovanni-time-series-adapter`

A `.netrc` file must also be located in the `test` directory of this repository.

## Import Requirements

In [ ]:
from harmony import Client, Collection, Request, Environment, WKT
import datetime as dt

## Set Default Parameters

`papermill` requires default values for parameters used on the workflow. In this case, `harmony_host_url`.

In [ ]:
harmony_host_url = 'https://harmony.uat.earthdata.nasa.gov'
# harmony_host_url = 'https://harmony.sit.earthdata.nasa.gov'
# harmony_host_url = 'http://localhost:3000'
# harmony_host_url = 'https://harmony.earthdata.nasa.gov'

### Identify Harmony environment (for easier reference)

In [ ]:
host_environment = {
    'http://localhost:3000': Environment.LOCAL,
    'https://harmony.sit.earthdata.nasa.gov': Environment.SIT,
    'https://harmony.uat.earthdata.nasa.gov': Environment.UAT,
    'https://harmony.earthdata.nasa.gov': Environment.PROD,
}


harmony_environment = host_environment.get(harmony_host_url)

if harmony_environment is not None:
    harmony_client = Client(env=harmony_environment)

### Identify Test Collection Ids and Expected Time Series URL Results

In [ ]:
non_prod_collection_id = Collection(id="C1240560361-GES_DISC")
non_prod_time_series_url = "https://api.giovanni.uat.earthdata.nasa.gov/proxy-timeseries?data=GPM_3IMERGHH_07_precipitation&location=%5B34.05%2C-118.25%5D&time=2020-01-06T12%3A00%3A00/2020-01-06T16%3A00%3A00"
prod_collection_id = Collection(id="C2723754847-GES_DISC")
prod_time_series_url = "https://api.giovanni.earthdata.nasa.gov/proxy-timeseries?data=GPM_3IMERGHH_07_precipitation&location=%5B34.05%2C-118.25%5D&time=2020-01-06T12%3A00%3A00/2020-01-06T16%3A00%3A00"


if harmony_environment == Environment.PROD:
    collection = prod_collection_id
    time_series_url = prod_time_series_url
else:
    collection = non_prod_collection_id
    time_series_url = non_prod_time_series_url

## Run Time Series Adapter Test 

### Make a Time Series Request

In [ ]:
request = Request(
    collection=collection,
    temporal={
        "start": dt.datetime(2020, 1, 6, 12, 0, 0, 0, tzinfo=dt.timezone.utc),
        "stop": dt.datetime(2020, 1, 6, 16, 0, 0, 0, tzinfo=dt.timezone.utc),
    },
    spatial=WKT("POINT(-118.25 34.05)"),
    variables=["GPM_3IMERGHH_07_precipitation"],
    format="text/csv",
    labels=["giovanni-tsa-rtests"],
)

job_id = harmony_client.submit(request)

### Confirm Request Results Exist and are Expected

In [ ]:
result_json = harmony_client.result_json(job_id)
time_series_url_result = result_json["links"][1]["href"]

assert (
    time_series_url_result == time_series_url
), "The Cloud Giovanni Time Series URL generated by Harmony does not match the expected URL"